In [ ]:
# ============================================================
# B3 - DETECCIÓN DE POSIBLE DATA LEAKAGE CON pHASH
# TRAIN vs TEST
# ============================================================

# 1. INSTALAR LIBRERÍAS
!pip install -q Pillow ImageHash openpyxl


# ============================================================
# 2. IMPORTAR LIBRERÍAS
# ============================================================

import os
import zipfile
import shutil
import pandas as pd

from PIL import Image, ImageOps, ImageDraw, ImageFont
import imagehash

from google.colab import files


# ============================================================
# 3. SUBIR ZIP DE ROBOFLOW
# ============================================================

print("Selecciona el ZIP descargado de Roboflow...")

uploaded = files.upload()

zip_name = list(uploaded.keys())[0]

print("\nZIP seleccionado:")
print(zip_name)


# ============================================================
# 4. DESCOMPRIMIR DATASET
# ============================================================

extract_path = "/content/dataset"

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

os.makedirs(extract_path)

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("\nDataset descomprimido en:")
print(extract_path)


# ============================================================
# 5. BUSCAR AUTOMÁTICAMENTE TRAIN Y TEST
# ============================================================

train_dir = None
test_dir = None

for root, dirs, files_in_dir in os.walk(extract_path):

    folder_name = os.path.basename(root).lower()

    if folder_name == "train":
        train_dir = root

    elif folder_name == "test":
        test_dir = root


print("\nTRAIN:")
print(train_dir)

print("\nTEST:")
print(test_dir)


if train_dir is None:
    raise Exception("No se encontró la carpeta TRAIN.")

if test_dir is None:
    raise Exception("No se encontró la carpeta TEST.")


# ============================================================
# 6. OBTENER TODAS LAS IMÁGENES
# ============================================================

image_extensions = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)


def get_images(folder):

    images = []

    for root, dirs, files_in_dir in os.walk(folder):

        for file in files_in_dir:

            if file.lower().endswith(image_extensions):

                images.append(
                    os.path.join(root, file)
                )

    return sorted(images)


train_images = get_images(train_dir)
test_images = get_images(test_dir)


print("\nNúmero de imágenes:")

print("TRAIN:", len(train_images))
print("TEST :", len(test_images))


# ============================================================
# 7. CALCULAR pHASH
# ============================================================

def calculate_phash(image_path):

    try:

        image = Image.open(image_path).convert("RGB")

        return imagehash.phash(image)

    except Exception as e:

        print(
            f"Error procesando {image_path}: {e}"
        )

        return None


print("\nCalculando pHash de TRAIN...")

train_hashes = {}

for i, image_path in enumerate(train_images):

    h = calculate_phash(image_path)

    if h is not None:

        train_hashes[image_path] = h

    if (i + 1) % 100 == 0:

        print(
            f"TRAIN procesadas: {i + 1}/{len(train_images)}"
        )


print("\nCalculando pHash de TEST...")

test_hashes = {}

for i, image_path in enumerate(test_images):

    h = calculate_phash(image_path)

    if h is not None:

        test_hashes[image_path] = h

    if (i + 1) % 100 == 0:

        print(
            f"TEST procesadas: {i + 1}/{len(test_images)}"
        )


print("\npHash calculados.")

print("TRAIN:", len(train_hashes))
print("TEST :", len(test_hashes))


# ============================================================
# 8. COMPARAR TRAIN VS TEST
# ============================================================

print("\nComparando TRAIN vs TEST...")

results = []

for test_path, test_hash in test_hashes.items():

    for train_path, train_hash in train_hashes.items():

        distance = test_hash - train_hash

        results.append({

            "test_image": test_path,

            "train_image": train_path,

            "phash_distance": distance

        })


df = pd.DataFrame(results)

df = df.sort_values(
    "phash_distance",
    ascending=True
).reset_index(drop=True)


print("\nComparaciones realizadas:")
print(len(df))


# ============================================================
# 9. CLASIFICAR SIMILITUD
# ============================================================

def classify_distance(distance):

    if distance <= 2:

        return "Casi idénticas"

    elif distance <= 5:

        return "Muy similares"

    elif distance <= 10:

        return "Similares"

    else:

        return "Probablemente diferentes"


df["similarity_category"] = (
    df["phash_distance"]
    .apply(classify_distance)
)


# ============================================================
# 10. FILTRAR PAREJAS SOSPECHOSAS
# ============================================================

# Umbral utilizado para revisión manual
PHASH_THRESHOLD = 5

suspicious = df[
    df["phash_distance"] <= PHASH_THRESHOLD
].copy()


print("\n========================================")
print("RESULTADOS")
print("========================================")

print(
    f"Parejas sospechosas (pHash <= {PHASH_THRESHOLD}): "
    f"{len(suspicious)}"
)


print("\nPrimeras parejas sospechosas:")

display(
    suspicious.head(20)
)


# ============================================================
# 11. CREAR CARPETA PARA LAS PAREJAS
# ============================================================

output_folder = "/content/B3_pHash_suspicious_pairs"

if os.path.exists(output_folder):

    shutil.rmtree(output_folder)

os.makedirs(output_folder)


# ============================================================
# 12. FUNCIÓN PARA CREAR IMAGEN LADO A LADO
# ============================================================

def create_side_by_side(
    train_path,
    test_path,
    distance,
    output_path
):

    try:

        # Abrir imágenes
        train_img = Image.open(train_path).convert("RGB")
        test_img = Image.open(test_path).convert("RGB")


        # Tamaño máximo
        max_height = 500


        def resize_image(img):

            ratio = max_height / img.height

            new_width = int(
                img.width * ratio
            )

            return img.resize(
                (new_width, max_height)
            )


        train_img = resize_image(train_img)
        test_img = resize_image(test_img)


        # Espacio superior para títulos
        header_height = 90

        total_width = (
            train_img.width +
            test_img.width
        )

        total_height = (
            max_height +
            header_height
        )


        canvas = Image.new(
            "RGB",
            (
                total_width,
                total_height
            ),
            "white"
        )


        # Colocar imágenes
        canvas.paste(
            train_img,
            (0, header_height)
        )

        canvas.paste(
            test_img,
            (
                train_img.width,
                header_height
            )
        )


        # Dibujar texto
        draw = ImageDraw.Draw(canvas)


        try:

            font = ImageFont.truetype(
                "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
                20
            )

            font_small = ImageFont.truetype(
                "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
                15
            )

        except:

            font = None
            font_small = None


        # Título
        draw.text(
            (10, 10),
            f"TRAIN | pHash: {distance}",
            fill="black",
            font=font
        )

        draw.text(
            (
                train_img.width + 10,
                10
            ),
            f"TEST | pHash: {distance}",
            fill="black",
            font=font
        )


        # Nombres de archivos
        draw.text(
            (10, 45),
            os.path.basename(train_path),
            fill="black",
            font=font_small
        )

        draw.text(
            (
                train_img.width + 10,
                45
            ),
            os.path.basename(test_path),
            fill="black",
            font=font_small
        )


        # Guardar
        canvas.save(
            output_path,
            quality=95
        )


    except Exception as e:

        print(
            f"Error creando comparación: {e}"
        )


# ============================================================
# 13. GENERAR LAS PAREJAS SOSPECHOSAS
# ============================================================

print("\nGenerando imágenes lado a lado...")


for index, row in suspicious.iterrows():

    train_path = row["train_image"]

    test_path = row["test_image"]

    distance = row["phash_distance"]


    filename = (
        f"pair_{index:04d}"
        f"_distance_{distance}.jpg"
    )


    output_path = os.path.join(
        output_folder,
        filename
    )


    create_side_by_side(
        train_path,
        test_path,
        distance,
        output_path
    )


print(
    "\nImágenes generadas:",
    len(os.listdir(output_folder))
)


# ============================================================
# 14. CREAR EXCEL FINAL
# ============================================================

excel_path = "/content/B3_pHash_results.xlsx"


# DataFrame completo
df_excel = df.copy()


# Agregar número de pareja sospechosa
df_excel["suspicious"] = (
    df_excel["phash_distance"]
    <= PHASH_THRESHOLD
)


df_excel.to_excel(
    excel_path,
    index=False
)


print("\nExcel creado:")
print(excel_path)


# ============================================================
# 15. CREAR ZIP DE LAS PAREJAS
# ============================================================

zip_output = "/content/B3_pHash_suspicious_pairs.zip"


if os.path.exists(zip_output):

    os.remove(zip_output)


shutil.make_archive(
    "/content/B3_pHash_suspicious_pairs",
    "zip",
    output_folder
)


print("\nZIP creado:")
print(zip_output)


# ============================================================
# 16. RESUMEN FINAL
# ============================================================

print("\n")
print("========================================")
print("          ANÁLISIS B3 FINALIZADO")
print("========================================")

print(f"TRAIN: {len(train_images)} imágenes")
print(f"TEST : {len(test_images)} imágenes")

print(
    f"Comparaciones: {len(df):,}"
)

print(
    f"Parejas pHash <= {PHASH_THRESHOLD}: "
    f"{len(suspicious)}"
)

print("\nArchivos generados:")

print("1. Excel:")
print(excel_path)

print("\n2. Carpeta:")
print(output_folder)

print("\n3. ZIP:")
print(zip_output)


# ============================================================
# 17. DESCARGAR EXCEL Y ZIP
# ============================================================

print("\nDescargando Excel...")

files.download(excel_path)

print("\nDescargando ZIP de parejas sospechosas...")

files.download(zip_output)

In [ ]:
# ============================================================
# AUDITORÍA COMPLETA DE DATA LEAKAGE
# TRAIN ↔ TEST
# TRAIN ↔ VALID
# VALID ↔ TEST
# ============================================================

import os
import pandas as pd
from PIL import Image
import imagehash


# ------------------------------------------------------------
# 1. DEFINIR CARPETAS
# ------------------------------------------------------------

train_dir = "/content/dataset/train"
valid_dir = "/content/dataset/valid"
test_dir  = "/content/dataset/test"

image_extensions = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)


# ------------------------------------------------------------
# 2. OBTENER IMÁGENES
# ------------------------------------------------------------

def get_images(folder):

    images = []

    for root, dirs, files in os.walk(folder):

        for file in files:

            if file.lower().endswith(image_extensions):

                images.append(
                    os.path.join(root, file)
                )

    return sorted(images)


train_images = get_images(train_dir)
valid_images = get_images(valid_dir)
test_images  = get_images(test_dir)


print("===================================")
print("IMÁGENES DEL DATASET")
print("===================================")

print("TRAIN:", len(train_images))
print("VALID:", len(valid_images))
print("TEST :", len(test_images))


# ------------------------------------------------------------
# 3. CALCULAR pHASH
# ------------------------------------------------------------

def calculate_phash(image_path):

    try:

        image = Image.open(image_path).convert("RGB")

        return imagehash.phash(image)

    except Exception as e:

        print("Error:", image_path, e)

        return None


def generate_hashes(images, name):

    hashes = {}

    print(f"\nCalculando pHash de {name}...")

    for i, image_path in enumerate(images):

        h = calculate_phash(image_path)

        if h is not None:

            hashes[image_path] = h

        if (i + 1) % 100 == 0:

            print(
                f"{name}: {i+1}/{len(images)}"
            )

    return hashes


train_hashes = generate_hashes(
    train_images,
    "TRAIN"
)

valid_hashes = generate_hashes(
    valid_images,
    "VALID"
)

test_hashes = generate_hashes(
    test_images,
    "TEST"
)


# ------------------------------------------------------------
# 4. FUNCIÓN DE COMPARACIÓN
# ------------------------------------------------------------

def compare_partitions(
    hashes_A,
    hashes_B,
    name_A,
    name_B,
    threshold=5
):

    results = []

    print(
        f"\nComparando {name_A} vs {name_B}..."
    )

    for path_A, hash_A in hashes_A.items():

        for path_B, hash_B in hashes_B.items():

            distance = hash_A - hash_B

            # Guardamos solamente candidatos
            if distance <= threshold:

                results.append({

                    f"{name_A.lower()}_image":
                        path_A,

                    f"{name_B.lower()}_image":
                        path_B,

                    "phash_distance":
                        distance

                })


    df = pd.DataFrame(results)

    if len(df) > 0:

        df = df.sort_values(
            "phash_distance"
        ).reset_index(drop=True)


    print(
        f"Candidatos pHash <= {threshold}:",
        len(df)
    )

    return df


# ------------------------------------------------------------
# 5. EJECUTAR LAS 3 COMPARACIONES
# ------------------------------------------------------------

THRESHOLD = 5


train_test = compare_partitions(
    train_hashes,
    test_hashes,
    "TRAIN",
    "TEST",
    THRESHOLD
)


train_valid = compare_partitions(
    train_hashes,
    valid_hashes,
    "TRAIN",
    "VALID",
    THRESHOLD
)


valid_test = compare_partitions(
    valid_hashes,
    test_hashes,
    "VALID",
    "TEST",
    THRESHOLD
)


# ------------------------------------------------------------
# 6. MOSTRAR RESUMEN
# ------------------------------------------------------------

print("\n")
print("===================================")
print("RESUMEN DE AUDITORÍA")
print("===================================")

print(
    "TRAIN ↔ TEST :",
    len(train_test),
    "candidatos"
)

print(
    "TRAIN ↔ VALID:",
    len(train_valid),
    "candidatos"
)

print(
    "VALID ↔ TEST :",
    len(valid_test),
    "candidatos"
)


# ------------------------------------------------------------
# 7. GUARDAR TODO EN UN EXCEL
# ------------------------------------------------------------

excel_path = (
    "/content/B3_complete_partition_audit.xlsx"
)


with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    train_test.to_excel(
        writer,
        sheet_name="TRAIN_TEST",
        index=False
    )

    train_valid.to_excel(
        writer,
        sheet_name="TRAIN_VALID",
        index=False
    )

    valid_test.to_excel(
        writer,
        sheet_name="VALID_TEST",
        index=False
    )


print("\nExcel creado:")
print(excel_path)


# ------------------------------------------------------------
# 8. DESCARGAR EXCEL
# ------------------------------------------------------------

from google.colab import files

files.download(excel_path)

In [ ]:
# ============================================================
# GENERAR AUDITORÍA VISUAL B3
# TRAIN-TEST / TRAIN-VALID / VALID-TEST
# Usa los DataFrames ya creados:
# train_test, train_valid, valid_test
# ============================================================

import os
import shutil
from PIL import Image, ImageDraw, ImageFont
from google.colab import files


# ============================================================
# 1. CARPETA PRINCIPAL
# ============================================================

base_output = "/content/B3_VISUAL_AUDIT"

if os.path.exists(base_output):
    shutil.rmtree(base_output)

os.makedirs(base_output)


# ============================================================
# 2. FUNCIÓN PARA CREAR COMPARACIÓN LADO A LADO
# ============================================================

def create_side_by_side(
    path_A,
    path_B,
    distance,
    label_A,
    label_B,
    output_path
):

    try:
        img_A = Image.open(path_A).convert("RGB")
        img_B = Image.open(path_B).convert("RGB")

        # Altura uniforme
        max_height = 500

        def resize_image(img):
            ratio = max_height / img.height
            new_width = int(img.width * ratio)

            return img.resize(
                (new_width, max_height)
            )

        img_A = resize_image(img_A)
        img_B = resize_image(img_B)

        # Espacio superior para información
        header_height = 110

        total_width = img_A.width + img_B.width
        total_height = max_height + header_height

        # Canvas blanco
        canvas = Image.new(
            "RGB",
            (total_width, total_height),
            "white"
        )

        # Pegar imágenes
        canvas.paste(
            img_A,
            (0, header_height)
        )

        canvas.paste(
            img_B,
            (img_A.width, header_height)
        )

        # Dibujar texto
        draw = ImageDraw.Draw(canvas)

        try:
            font_title = ImageFont.truetype(
                "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
                20
            )

            font_small = ImageFont.truetype(
                "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
                13
            )

        except:
            font_title = None
            font_small = None

        # Encabezados
        draw.text(
            (10, 10),
            label_A,
            fill="black",
            font=font_title
        )

        draw.text(
            (img_A.width + 10, 10),
            label_B,
            fill="black",
            font=font_title
        )

        # Distancia pHash
        draw.text(
            (10, 40),
            f"pHash distance = {distance}",
            fill="black",
            font=font_title
        )

        # Nombre archivo A
        name_A = os.path.basename(path_A)

        draw.text(
            (10, 75),
            name_A[:55],
            fill="black",
            font=font_small
        )

        # Nombre archivo B
        name_B = os.path.basename(path_B)

        draw.text(
            (img_A.width + 10, 75),
            name_B[:55],
            fill="black",
            font=font_small
        )

        # Línea divisoria
        draw.line(
            (
                img_A.width,
                0,
                img_A.width,
                total_height
            ),
            fill="black",
            width=3
        )

        # Guardar
        canvas.save(
            output_path,
            quality=95
        )

    except Exception as e:
        print(
            f"ERROR con {path_A} / {path_B}: {e}"
        )


# ============================================================
# 3. FUNCIÓN PARA PROCESAR CADA COMPARACIÓN
# ============================================================

def generate_visual_folder(
    dataframe,
    comparison_name,
    column_A,
    column_B,
    label_A,
    label_B
):

    folder = os.path.join(
        base_output,
        comparison_name
    )

    os.makedirs(
        folder,
        exist_ok=True
    )

    print(
        f"\nGenerando {comparison_name}..."
    )

    generated = 0

    for i, row in dataframe.iterrows():

        path_A = row[column_A]
        path_B = row[column_B]
        distance = int(row["phash_distance"])

        filename = (
            f"{i+1:03d}"
            f"_pHash_{distance}.jpg"
        )

        output_path = os.path.join(
            folder,
            filename
        )

        create_side_by_side(
            path_A,
            path_B,
            distance,
            label_A,
            label_B,
            output_path
        )

        generated += 1

    print(
        f"{comparison_name}: "
        f"{generated} imágenes generadas."
    )


# ============================================================
# 4. TRAIN vs TEST
# ============================================================

generate_visual_folder(
    train_test,
    "TRAIN_TEST",
    "train_image",
    "test_image",
    "TRAIN",
    "TEST"
)


# ============================================================
# 5. TRAIN vs VALID
# ============================================================

generate_visual_folder(
    train_valid,
    "TRAIN_VALID",
    "train_image",
    "valid_image",
    "TRAIN",
    "VALID"
)


# ============================================================
# 6. VALID vs TEST
# ============================================================

generate_visual_folder(
    valid_test,
    "VALID_TEST",
    "valid_image",
    "test_image",
    "VALID",
    "TEST"
)


# ============================================================
# 7. CONTAR ARCHIVOS
# ============================================================

print("\n========================================")
print("RESUMEN")
print("========================================")

for folder_name in [
    "TRAIN_TEST",
    "TRAIN_VALID",
    "VALID_TEST"
]:

    folder_path = os.path.join(
        base_output,
        folder_name
    )

    number = len(
        os.listdir(folder_path)
    )

    print(
        f"{folder_name}: {number} comparaciones"
    )


# ============================================================
# 8. CREAR ZIP
# ============================================================

zip_base = "/content/B3_VISUAL_AUDIT"

zip_path = zip_base + ".zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(
    zip_base,
    "zip",
    base_output
)


print("\n========================================")
print("ZIP CREADO")
print("========================================")

print(zip_path)


# ============================================================
# 9. DESCARGAR ZIP
# ============================================================

files.download(zip_path)

In [ ]:
# ================================================================
# B3 - COMPLETE DATA LEAKAGE AUDIT
# pHash Cross-Partition Analysis
# TRAIN ↔ TEST | TRAIN ↔ VALID | VALID ↔ TEST
# ================================================================

!pip install -q Pillow ImageHash pandas openpyxl

import os
import zipfile
import shutil
import pandas as pd

from PIL import Image, ImageDraw, ImageFont
import imagehash
from google.colab import files


# ================================================================
# CONFIGURACIÓN
# ================================================================

PHASH_THRESHOLD = 5

DATASET_FOLDER = "/content/dataset"
OUTPUT_FOLDER = "/content/B3_VISUAL_AUDIT"

EXCEL_PATH = "/content/B3_complete_partition_audit.xlsx"
ZIP_OUTPUT = "/content/B3_VISUAL_AUDIT.zip"

IMAGE_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)


# ================================================================
# 1. SUBIR ZIP DE ROBOFLOW
# ================================================================

print("================================================")
print("B3 - COMPLETE DATA LEAKAGE AUDIT")
print("================================================")

print("\nSelecciona el ZIP de Roboflow...\n")

uploaded = files.upload()

zip_name = list(uploaded.keys())[0]

print("\nZIP seleccionado:")
print(zip_name)


# ================================================================
# 2. LIMPIAR EJECUCIÓN ANTERIOR
# ================================================================

if os.path.exists(DATASET_FOLDER):
    shutil.rmtree(DATASET_FOLDER)

if os.path.exists(OUTPUT_FOLDER):
    shutil.rmtree(OUTPUT_FOLDER)

if os.path.exists(EXCEL_PATH):
    os.remove(EXCEL_PATH)

if os.path.exists(ZIP_OUTPUT):
    os.remove(ZIP_OUTPUT)

os.makedirs(DATASET_FOLDER)
os.makedirs(OUTPUT_FOLDER)


# ================================================================
# 3. DESCOMPRIMIR DATASET
# ================================================================

print("\nDescomprimiendo dataset...")

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(DATASET_FOLDER)

print("Dataset descomprimido.")


# ================================================================
# 4. BUSCAR TRAIN / VALID / TEST AUTOMÁTICAMENTE
# ================================================================

train_dir = None
valid_dir = None
test_dir = None

for root, dirs, filenames in os.walk(DATASET_FOLDER):

    folder_name = os.path.basename(root).lower()

    if folder_name == "train":
        train_dir = root

    elif folder_name in ["valid", "val", "validation"]:
        valid_dir = root

    elif folder_name == "test":
        test_dir = root


if train_dir is None:
    raise Exception("No se encontró TRAIN.")

if valid_dir is None:
    raise Exception("No se encontró VALID.")

if test_dir is None:
    raise Exception("No se encontró TEST.")


print("\n========================================")
print("PARTICIONES ENCONTRADAS")
print("========================================")

print("TRAIN:", train_dir)
print("VALID:", valid_dir)
print("TEST :", test_dir)


# ================================================================
# 5. OBTENER IMÁGENES
# ================================================================

def get_images(folder):

    image_list = []

    for root, dirs, filenames in os.walk(folder):

        for filename in filenames:

            if filename.lower().endswith(IMAGE_EXTENSIONS):

                image_list.append(
                    os.path.join(root, filename)
                )

    return sorted(image_list)


train_images = get_images(train_dir)
valid_images = get_images(valid_dir)
test_images = get_images(test_dir)


total_images = (
    len(train_images) +
    len(valid_images) +
    len(test_images)
)


print("\n========================================")
print("DATASET")
print("========================================")

print(f"TRAIN: {len(train_images)}")
print(f"VALID: {len(valid_images)}")
print(f"TEST : {len(test_images)}")

print("----------------------------------------")

print(f"TOTAL: {total_images}")


# ================================================================
# 6. CALCULAR PORCENTAJES REALES
# ================================================================

train_percentage = (
    len(train_images) /
    total_images * 100
)

valid_percentage = (
    len(valid_images) /
    total_images * 100
)

test_percentage = (
    len(test_images) /
    total_images * 100
)


print("\nDistribución:")

print(
    f"TRAIN: {train_percentage:.2f}%"
)

print(
    f"VALID: {valid_percentage:.2f}%"
)

print(
    f"TEST : {test_percentage:.2f}%"
)


# ================================================================
# 7. CALCULAR pHASH
# ================================================================

def calculate_phash(image_path):

    try:

        with Image.open(image_path) as image:

            image = image.convert("RGB")

            return imagehash.phash(image)

    except Exception as e:

        print(
            f"ERROR: {image_path}: {e}"
        )

        return None


def generate_hashes(images, name):

    hashes = {}

    print(
        f"\nCalculando pHash de {name}..."
    )

    for i, image_path in enumerate(images):

        h = calculate_phash(image_path)

        if h is not None:

            hashes[image_path] = h

        if (i + 1) % 100 == 0:

            print(
                f"{name}: "
                f"{i+1}/{len(images)}"
            )

    print(
        f"{name} completado: "
        f"{len(hashes)} hashes"
    )

    return hashes


train_hashes = generate_hashes(
    train_images,
    "TRAIN"
)

valid_hashes = generate_hashes(
    valid_images,
    "VALID"
)

test_hashes = generate_hashes(
    test_images,
    "TEST"
)


# ================================================================
# 8. COMPARAR PARTICIONES
# ================================================================

def compare_partitions(
    hashes_A,
    hashes_B,
    name_A,
    name_B
):

    results = []

    total_comparisons = (
        len(hashes_A) *
        len(hashes_B)
    )

    print("\n========================================")

    print(
        f"Comparando {name_A} ↔ {name_B}"
    )

    print(
        f"Comparaciones totales: "
        f"{total_comparisons:,}"
    )

    print("========================================")


    for path_A, hash_A in hashes_A.items():

        for path_B, hash_B in hashes_B.items():

            distance = hash_A - hash_B

            if distance <= PHASH_THRESHOLD:

                results.append({

                    "partition_A":
                        name_A,

                    "image_A":
                        path_A,

                    "filename_A":
                        os.path.basename(path_A),

                    "partition_B":
                        name_B,

                    "image_B":
                        path_B,

                    "filename_B":
                        os.path.basename(path_B),

                    "phash_distance":
                        distance

                })


    df = pd.DataFrame(results)


    if not df.empty:

        df = df.sort_values(
            "phash_distance"
        ).reset_index(drop=True)


    print(
        f"Candidatos pHash <= "
        f"{PHASH_THRESHOLD}: {len(df)}"
    )


    if not df.empty:

        print("\nDistribución:")

        print(
            df["phash_distance"]
            .value_counts()
            .sort_index()
        )


    return df, total_comparisons


# ================================================================
# 9. EJECUTAR LAS TRES COMPARACIONES
# ================================================================

train_test, comp_train_test = (
    compare_partitions(
        train_hashes,
        test_hashes,
        "TRAIN",
        "TEST"
    )
)


train_valid, comp_train_valid = (
    compare_partitions(
        train_hashes,
        valid_hashes,
        "TRAIN",
        "VALID"
    )
)


valid_test, comp_valid_test = (
    compare_partitions(
        valid_hashes,
        test_hashes,
        "VALID",
        "TEST"
    )
)


# ================================================================
# 10. CREAR IMAGEN LADO A LADO
# ================================================================

def create_side_by_side(
    path_A,
    path_B,
    distance,
    label_A,
    label_B,
    output_path
):

    try:

        img_A = Image.open(
            path_A
        ).convert("RGB")

        img_B = Image.open(
            path_B
        ).convert("RGB")


        max_height = 500


        def resize_image(img):

            ratio = (
                max_height /
                img.height
            )

            new_width = int(
                img.width *
                ratio
            )

            return img.resize(
                (
                    new_width,
                    max_height
                )
            )


        img_A = resize_image(img_A)
        img_B = resize_image(img_B)


        header_height = 120

        total_width = (
            img_A.width +
            img_B.width
        )

        total_height = (
            max_height +
            header_height
        )


        canvas = Image.new(
            "RGB",
            (
                total_width,
                total_height
            ),
            "white"
        )


        canvas.paste(
            img_A,
            (0, header_height)
        )

        canvas.paste(
            img_B,
            (
                img_A.width,
                header_height
            )
        )


        draw = ImageDraw.Draw(
            canvas
        )


        try:

            font_title = (
                ImageFont.truetype(
                    "/usr/share/fonts/"
                    "truetype/dejavu/"
                    "DejaVuSans-Bold.ttf",
                    20
                )
            )

            font_small = (
                ImageFont.truetype(
                    "/usr/share/fonts/"
                    "truetype/dejavu/"
                    "DejaVuSans.ttf",
                    12
                )
            )

        except:

            font_title = None
            font_small = None


        # Títulos
        draw.text(
            (10, 10),
            label_A,
            fill="black",
            font=font_title
        )

        draw.text(
            (
                img_A.width + 10,
                10
            ),
            label_B,
            fill="black",
            font=font_title
        )


        # pHash
        draw.text(
            (10, 42),
            f"pHash distance = "
            f"{distance}",
            fill="black",
            font=font_title
        )


        # Nombres
        draw.text(
            (10, 80),
            os.path.basename(
                path_A
            )[:60],
            fill="black",
            font=font_small
        )

        draw.text(
            (
                img_A.width + 10,
                80
            ),
            os.path.basename(
                path_B
            )[:60],
            fill="black",
            font=font_small
        )


        # Línea divisoria
        draw.line(
            (
                img_A.width,
                0,
                img_A.width,
                total_height
            ),
            fill="black",
            width=3
        )


        canvas.save(
            output_path,
            quality=95
        )


    except Exception as e:

        print(
            f"Error creando imagen: {e}"
        )


# ================================================================
# 11. GENERAR CARPETAS VISUALES
# ================================================================

def generate_visual_folder(
    dataframe,
    comparison_name
):

    folder = os.path.join(
        OUTPUT_FOLDER,
        comparison_name
    )

    os.makedirs(
        folder,
        exist_ok=True
    )


    if dataframe.empty:

        print(
            f"\n{comparison_name}: "
            f"0 candidatos."
        )

        return


    print(
        f"\nGenerando auditoría visual "
        f"{comparison_name}..."
    )


    for i, row in dataframe.iterrows():

        distance = int(
            row["phash_distance"]
        )

        filename = (
            f"{i+1:03d}"
            f"_pHash_{distance}.jpg"
        )

        output_path = os.path.join(
            folder,
            filename
        )


        create_side_by_side(

            row["image_A"],

            row["image_B"],

            distance,

            row["partition_A"],

            row["partition_B"],

            output_path
        )


    print(
        f"{comparison_name}: "
        f"{len(dataframe)} "
        f"comparaciones visuales."
    )


generate_visual_folder(
    train_test,
    "TRAIN_TEST"
)

generate_visual_folder(
    train_valid,
    "TRAIN_VALID"
)

generate_visual_folder(
    valid_test,
    "VALID_TEST"
)


# ================================================================
# 12. CREAR RESUMEN
# ================================================================

summary = pd.DataFrame({

    "Comparison": [
        "TRAIN-TEST",
        "TRAIN-VALID",
        "VALID-TEST"
    ],

    "Total comparisons": [
        comp_train_test,
        comp_train_valid,
        comp_valid_test
    ],

    f"Candidates pHash <= {PHASH_THRESHOLD}": [
        len(train_test),
        len(train_valid),
        len(valid_test)
    ]

})


dataset_summary = pd.DataFrame({

    "Partition": [
        "TRAIN",
        "VALID",
        "TEST",
        "TOTAL"
    ],

    "Images": [
        len(train_images),
        len(valid_images),
        len(test_images),
        total_images
    ],

    "Percentage": [
        train_percentage,
        valid_percentage,
        test_percentage,
        100
    ]

})


# ================================================================
# 13. CREAR EXCEL
# ================================================================

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:


    dataset_summary.to_excel(
        writer,
        sheet_name="DATASET_SUMMARY",
        index=False
    )


    summary.to_excel(
        writer,
        sheet_name="AUDIT_SUMMARY",
        index=False
    )


    train_test.to_excel(
        writer,
        sheet_name="TRAIN_TEST",
        index=False
    )


    train_valid.to_excel(
        writer,
        sheet_name="TRAIN_VALID",
        index=False
    )


    valid_test.to_excel(
        writer,
        sheet_name="VALID_TEST",
        index=False
    )


print("\nExcel generado:")
print(EXCEL_PATH)


# ================================================================
# 14. CREAR ZIP VISUAL
# ================================================================

shutil.make_archive(
    "/content/B3_VISUAL_AUDIT",
    "zip",
    OUTPUT_FOLDER
)


print("\nZIP generado:")
print(ZIP_OUTPUT)


# ================================================================
# 15. RESULTADO FINAL
# ================================================================

print("\n")
print("================================================")
print("        B3 AUDIT COMPLETED")
print("================================================")

print("\nDATASET:")

print(
    f"TRAIN = {len(train_images)} "
    f"({train_percentage:.2f}%)"
)

print(
    f"VALID = {len(valid_images)} "
    f"({valid_percentage:.2f}%)"
)

print(
    f"TEST  = {len(test_images)} "
    f"({test_percentage:.2f}%)"
)


print("\nCANDIDATOS:")

print(
    f"TRAIN ↔ TEST  = "
    f"{len(train_test)}"
)

print(
    f"TRAIN ↔ VALID = "
    f"{len(train_valid)}"
)

print(
    f"VALID ↔ TEST  = "
    f"{len(valid_test)}"
)


total_candidates = (
    len(train_test) +
    len(train_valid) +
    len(valid_test)
)


print(
    f"\nTOTAL CANDIDATOS = "
    f"{total_candidates}"
)


print("\nArchivos:")

print(EXCEL_PATH)
print(ZIP_OUTPUT)


# ================================================================
# 16. DESCARGAR
# ================================================================

print("\nDescargando Excel...")

files.download(
    EXCEL_PATH
)


print(
    "\nDescargando ZIP "
    "con comparaciones visuales..."
)

files.download(
    ZIP_OUTPUT
)